# Chapter 6.8: Multimodal Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** multimodal recommendation architectures combining text, visual, and audio features
2. **Implement** CLIP-style visual-text alignment for recommendation
3. **Build** visual feature extractors (simulated CNN/ViT) for item embeddings
4. **Compare** early, late, and cross-attention fusion strategies
5. **Handle** temporal visual features for video recommendation
6. **Evaluate** multimodal vs single-modal recommendation quality

## Prerequisites

- Understanding of collaborative filtering (Part 1)
- Familiarity with CNNs and attention mechanisms
- Chapter 6.5 (LLM as feature encoder)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part6/chapter_6.8_multimodal_rec.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/rec_system/main/notebooks/part6/chapter_6.8_multimodal_rec.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cpu')
print(f'Using device: {device}')

## 1. Multimodal Recommendation Overview

Items in real recommender systems have multiple modalities:

| Domain | Text | Visual | Audio | Other |
|--------|------|--------|-------|---------|
| E-commerce | Title, reviews | Product images | — | Price, category |
| Video | Title, description | Thumbnails, frames | Dialog, music | Watch time |
| Music | Lyrics, bio | Album art | Audio spectrogram | Genre tags |
| News | Headline, body | Article images | — | Publish time |

### CLIP for Recommendation

**CLIP** (Radford et al., 2021, OpenAI) learns aligned visual-text representations through contrastive learning. For recommendation:

$$\text{sim}(\text{text}_i, \text{image}_i) > \text{sim}(\text{text}_i, \text{image}_j) \quad \forall j \neq i$$

This alignment allows:
- Cross-modal search ("find items like this image")
- Multimodal item embeddings
- Zero-shot item categorization

> **💡 Concept:** CLIP-style models learn a shared embedding space where text and images are comparable. This is invaluable for recommendation because users may express preferences in text ("I want something blue and modern") while items are represented visually.

In [ ]:
# Generate synthetic multimodal item data
n_items = 500
n_users = 1000

text_dim = 64
visual_dim = 128
audio_dim = 32

n_categories = 8
category_names = ['Electronics', 'Fashion', 'Home', 'Sports', 'Books', 'Music', 'Food', 'Toys']
item_categories = np.random.randint(0, n_categories, n_items)

# Generate modality-specific features with shared category structure
cat_text_centers = np.random.randn(n_categories, text_dim) * 1.0
cat_visual_centers = np.random.randn(n_categories, visual_dim) * 1.0
cat_audio_centers = np.random.randn(n_categories, audio_dim) * 1.0

text_features = np.zeros((n_items, text_dim), dtype=np.float32)
visual_features = np.zeros((n_items, visual_dim), dtype=np.float32)
audio_features = np.zeros((n_items, audio_dim), dtype=np.float32)

for i in range(n_items):
    cat = item_categories[i]
    text_features[i] = cat_text_centers[cat] + np.random.randn(text_dim) * 0.3
    visual_features[i] = cat_visual_centers[cat] + np.random.randn(visual_dim) * 0.4
    audio_features[i] = cat_audio_centers[cat] + np.random.randn(audio_dim) * 0.3

# Generate user preferences and interactions
user_cat_prefs = np.random.randn(n_users, n_categories) * 0.5
# Some users prefer visual items, others text-based
user_modality_weight = np.random.dirichlet([2, 2, 1], size=n_users)  # text, visual, audio

# Generate click data
n_samples = 40000
user_ids = np.random.randint(0, n_users, n_samples)
item_ids = np.random.randint(0, n_items, n_samples)

# Labels based on category match + modality preferences
logits = np.array([
    user_cat_prefs[u, item_categories[i]] +
    user_modality_weight[u, 0] * np.dot(text_features[i][:8], np.random.randn(8)) * 0.1 +
    user_modality_weight[u, 1] * np.dot(visual_features[i][:8], np.random.randn(8)) * 0.1
    for u, i in zip(user_ids, item_ids)
])
labels = (logits + np.random.randn(n_samples) * 0.5 > 0).astype(np.float32)

split = int(0.8 * n_samples)
train_idx = np.arange(split)
test_idx = np.arange(split, n_samples)

print(f'Items: {n_items} ({n_categories} categories)')
print(f'Users: {n_users}')
print(f'Text dim: {text_dim}, Visual dim: {visual_dim}, Audio dim: {audio_dim}')
print(f'Samples: {n_samples} (positive rate: {labels.mean():.3f})')

## 2. CLIP-Style Alignment for Recommendation

In [ ]:
class CLIPAligner(nn.Module):
    """CLIP-style contrastive alignment between text and visual features.
    
    Reference: Radford et al., "Learning Transferable Visual Models From
    Natural Language Supervision", ICML 2021.
    """
    
    def __init__(self, text_dim, visual_dim, proj_dim=32, temperature=0.07):
        super().__init__()
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )
        self.visual_proj = nn.Sequential(
            nn.Linear(visual_dim, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )
        self.temperature = nn.Parameter(torch.tensor(temperature))
    
    def forward(self, text_feat, visual_feat):
        text_emb = F.normalize(self.text_proj(text_feat), dim=-1)
        visual_emb = F.normalize(self.visual_proj(visual_feat), dim=-1)
        return text_emb, visual_emb
    
    def contrastive_loss(self, text_feat, visual_feat):
        text_emb, visual_emb = self.forward(text_feat, visual_feat)
        logits = text_emb @ visual_emb.T / self.temperature.abs()
        labels = torch.arange(len(logits), device=logits.device)
        loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
        return loss


# Train CLIP-style alignment
clip = CLIPAligner(text_dim, visual_dim, proj_dim=32)
optimizer = torch.optim.Adam(clip.parameters(), lr=1e-3)

text_t = torch.FloatTensor(text_features)
visual_t = torch.FloatTensor(visual_features)

clip_losses = []
for epoch in range(100):
    idx = np.random.choice(n_items, 128, replace=False)
    loss = clip.contrastive_loss(text_t[idx], visual_t[idx])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    clip_losses.append(loss.item())
    if (epoch + 1) % 25 == 0:
        print(f'CLIP Epoch {epoch+1} | Loss: {loss.item():.4f}')

# Evaluate alignment quality
clip.eval()
with torch.no_grad():
    t_emb, v_emb = clip(text_t, visual_t)
    sim_matrix = (t_emb @ v_emb.T).cpu().numpy()

# Check retrieval accuracy: can text find its own image?
correct = 0
for i in range(n_items):
    if np.argmax(sim_matrix[i]) == i:
        correct += 1
print(f'\nText-to-Image retrieval accuracy: {correct/n_items:.2%}')

# Same-category retrieval
top5_same_cat = 0
for i in range(n_items):
    top5 = np.argsort(-sim_matrix[i])[:5]
    same_cat = sum(1 for j in top5 if item_categories[j] == item_categories[i])
    top5_same_cat += same_cat
print(f'Same-category in top-5 retrieval: {top5_same_cat / (n_items * 5):.2%}')

## 3. Fusion Strategies

How to combine features from multiple modalities:

### Early Fusion
Concatenate raw features before processing:
$$\mathbf{h} = f([\mathbf{x}_{\text{text}}; \mathbf{x}_{\text{visual}}; \mathbf{x}_{\text{audio}}])$$

### Late Fusion
Process each modality separately, combine at the end:
$$\mathbf{h} = g(f_{\text{text}}(\mathbf{x}_{\text{text}}), f_{\text{visual}}(\mathbf{x}_{\text{visual}}), f_{\text{audio}}(\mathbf{x}_{\text{audio}}))$$

### Cross-Attention Fusion
Modalities attend to each other:
$$\mathbf{h}_{\text{text}}' = \text{CrossAttn}(\mathbf{h}_{\text{text}}, \mathbf{h}_{\text{visual}})$$

> **⚠️ Common Pitfall:** Early fusion can be dominated by the higher-dimensional modality. Always normalize features from different modalities before concatenation.

In [ ]:
class EarlyFusionRec(nn.Module):
    """Early fusion: concatenate all modality features."""
    
    def __init__(self, n_users, text_dim, visual_dim, embed_dim=16, hidden_dim=128):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, embed_dim)
        total_dim = embed_dim + text_dim + visual_dim
        self.mlp = nn.Sequential(
            nn.Linear(total_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, user_ids, text_feat, visual_feat):
        u = self.user_embed(user_ids)
        # Normalize modalities
        text_norm = F.normalize(text_feat, dim=-1)
        visual_norm = F.normalize(visual_feat, dim=-1)
        x = torch.cat([u, text_norm, visual_norm], dim=-1)
        return self.mlp(x).squeeze(-1)


class LateFusionRec(nn.Module):
    """Late fusion: separate encoders per modality, combine outputs."""
    
    def __init__(self, n_users, text_dim, visual_dim, embed_dim=16, modal_dim=32):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, modal_dim), nn.ReLU()
        )
        self.visual_encoder = nn.Sequential(
            nn.Linear(visual_dim, modal_dim), nn.ReLU()
        )
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim + modal_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    
    def forward(self, user_ids, text_feat, visual_feat):
        u = self.user_embed(user_ids)
        t = self.text_encoder(text_feat)
        v = self.visual_encoder(visual_feat)
        x = torch.cat([u, t, v], dim=-1)
        return self.fusion(x).squeeze(-1)


class CrossAttentionFusionRec(nn.Module):
    """Cross-attention fusion: modalities attend to each other."""
    
    def __init__(self, n_users, text_dim, visual_dim, embed_dim=16, modal_dim=32):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.text_proj = nn.Linear(text_dim, modal_dim)
        self.visual_proj = nn.Linear(visual_dim, modal_dim)
        
        # Cross-attention: text attends to visual
        self.cross_attn = nn.MultiheadAttention(modal_dim, num_heads=4, batch_first=True)
        
        self.output = nn.Sequential(
            nn.Linear(embed_dim + modal_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    
    def forward(self, user_ids, text_feat, visual_feat):
        u = self.user_embed(user_ids)
        t = self.text_proj(text_feat).unsqueeze(1)  # (B, 1, D)
        v = self.visual_proj(visual_feat).unsqueeze(1)  # (B, 1, D)
        
        # Cross attention: text query, visual key/value
        t_attended, _ = self.cross_attn(t, v, v)
        
        t_out = t_attended.squeeze(1)
        v_out = v.squeeze(1)
        x = torch.cat([u, t_out, v_out], dim=-1)
        return self.output(x).squeeze(-1)


print('Fusion models defined.')

In [ ]:
# Train and compare fusion strategies
from sklearn.metrics import roc_auc_score

def train_multimodal(model, n_epochs=20, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    
    train_u = torch.LongTensor(user_ids[train_idx])
    train_t = torch.FloatTensor(text_features[item_ids[train_idx]])
    train_v = torch.FloatTensor(visual_features[item_ids[train_idx]])
    train_y = torch.FloatTensor(labels[train_idx])
    
    test_u = torch.LongTensor(user_ids[test_idx])
    test_t = torch.FloatTensor(text_features[item_ids[test_idx]])
    test_v = torch.FloatTensor(visual_features[item_ids[test_idx]])
    test_y = torch.FloatTensor(labels[test_idx])
    
    dataset = TensorDataset(train_u, train_t, train_v, train_y)
    loader = DataLoader(dataset, batch_size=512, shuffle=True)
    
    aucs = []
    for epoch in range(n_epochs):
        model.train()
        for u_b, t_b, v_b, y_b in loader:
            logits = model(u_b, t_b, v_b)
            loss = F.binary_cross_entropy_with_logits(logits, y_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                preds = torch.sigmoid(model(test_u, test_t, test_v)).numpy()
                auc = roc_auc_score(test_y.numpy(), preds)
                aucs.append(auc)
    
    return aucs


# Train all fusion strategies
fusion_results = {}

for name, ModelClass in [('Early Fusion', EarlyFusionRec),
                          ('Late Fusion', LateFusionRec),
                          ('Cross-Attention', CrossAttentionFusionRec)]:
    torch.manual_seed(42)
    model = ModelClass(n_users, text_dim, visual_dim)
    aucs = train_multimodal(model, n_epochs=20)
    fusion_results[name] = aucs
    print(f'{name:18s} | Final AUC: {aucs[-1]:.4f}')

# Plot comparison
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
eval_epochs = list(range(5, 21, 5))
for name, aucs in fusion_results.items():
    ax.plot(eval_epochs, aucs, '-o', label=name, markersize=6)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('AUC', fontsize=12)
ax.set_title('Multimodal Fusion Strategy Comparison', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Visual Features for Recommendation

In practice, visual features come from:
- **CNN features**: ResNet, EfficientNet (spatial features from images)
- **ViT features**: Vision Transformer (patch-based features)
- **Video features**: Temporal aggregation of frame features

Let's simulate extracting and using visual features.

In [ ]:
class SimulatedVisualEncoder(nn.Module):
    """Simulates a CNN/ViT feature extractor."""
    
    def __init__(self, raw_dim=256, output_dim=64):
        super().__init__()
        # Simulates ResNet-like feature extraction
        self.backbone = nn.Sequential(
            nn.Linear(raw_dim, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim),
            nn.ReLU()
        )
        # Global average pooling equivalent
        self.pool = nn.AdaptiveAvgPool1d(1)
    
    def forward(self, x):
        return self.backbone(x)


class TemporalVideoEncoder(nn.Module):
    """Encodes a sequence of video frame features."""
    
    def __init__(self, frame_dim=64, hidden_dim=32, n_frames=5):
        super().__init__()
        self.frame_proj = nn.Linear(frame_dim, hidden_dim)
        # Temporal attention
        self.temporal_attn = nn.Sequential(
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, frame_features):
        """frame_features: (batch, n_frames, frame_dim)"""
        h = self.frame_proj(frame_features)  # (batch, n_frames, hidden)
        attn_weights = F.softmax(self.temporal_attn(h), dim=1)  # (batch, n_frames, 1)
        pooled = (h * attn_weights).sum(dim=1)  # (batch, hidden)
        return pooled, attn_weights.squeeze(-1)


# Simulate video data: each item has 5 "frames"
n_frames = 5
frame_dim = 64
video_features = np.zeros((n_items, n_frames, frame_dim), dtype=np.float32)
for i in range(n_items):
    base = cat_visual_centers[item_categories[i]][:frame_dim]
    for f in range(n_frames):
        # Temporal variation within video
        video_features[i, f] = base + np.random.randn(frame_dim) * (0.2 + 0.1 * f)

# Test temporal encoder
temp_encoder = TemporalVideoEncoder(frame_dim=frame_dim, hidden_dim=32)
sample_video = torch.FloatTensor(video_features[:5])
pooled, attn_weights = temp_encoder(sample_video)

print(f'Video features shape: {video_features.shape}')
print(f'Pooled output shape: {pooled.shape}')
print(f'Temporal attention weights (item 0): {attn_weights[0].detach().numpy()}')

# Visualize temporal attention
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for i in range(5):
    ax.plot(range(n_frames), attn_weights[i].detach().numpy(), '-o', label=f'Item {i}', markersize=6)

ax.set_xlabel('Frame Index', fontsize=12)
ax.set_ylabel('Attention Weight', fontsize=12)
ax.set_title('Temporal Attention over Video Frames', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Modality Ablation Study

In [ ]:
class SingleModalRec(nn.Module):
    """Recommendation with a single modality."""
    def __init__(self, n_users, feat_dim, embed_dim=16, hidden_dim=64):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim + feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, user_ids, feat):
        u = self.user_embed(user_ids)
        x = torch.cat([u, F.normalize(feat, dim=-1)], dim=-1)
        return self.mlp(x).squeeze(-1)


def train_single_modal(model, user_ids_arr, item_feats, labels_arr, train_idx, test_idx, n_epochs=20):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    
    train_u = torch.LongTensor(user_ids_arr[train_idx])
    train_f = torch.FloatTensor(item_feats[item_ids[train_idx]])
    train_y = torch.FloatTensor(labels_arr[train_idx])
    
    test_u = torch.LongTensor(user_ids_arr[test_idx])
    test_f = torch.FloatTensor(item_feats[item_ids[test_idx]])
    test_y = torch.FloatTensor(labels_arr[test_idx])
    
    dataset = TensorDataset(train_u, train_f, train_y)
    loader = DataLoader(dataset, batch_size=512, shuffle=True)
    
    for epoch in range(n_epochs):
        model.train()
        for u_b, f_b, y_b in loader:
            logits = model(u_b, f_b)
            loss = F.binary_cross_entropy_with_logits(logits, y_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    model.eval()
    with torch.no_grad():
        preds = torch.sigmoid(model(test_u, test_f)).numpy()
        return roc_auc_score(test_y.numpy(), preds)


# Ablation study
ablation_results = {}

for modal_name, feats in [('Text Only', text_features),
                            ('Visual Only', visual_features),
                            ('Audio Only', audio_features)]:
    torch.manual_seed(42)
    model = SingleModalRec(n_users, feats.shape[1])
    auc = train_single_modal(model, user_ids, feats, labels, train_idx, test_idx)
    ablation_results[modal_name] = auc

# Add multimodal results
for name, aucs in fusion_results.items():
    ablation_results[name] = aucs[-1]

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
names = list(ablation_results.keys())
scores = [ablation_results[n] for n in names]
colors = ['#ff9999', '#ffcc99', '#99ccff'] + ['#66b266', '#339933', '#006600']

bars = ax.bar(range(len(names)), scores, color=colors[:len(names)])
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_ylabel('AUC')
ax.set_title('Modality Ablation: Single vs Multimodal Recommendation')
ax.grid(True, alpha=0.3, axis='y')

for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{score:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

> **🔑 Pro Tip:** Multimodal models consistently outperform single-modal ones, but the improvement varies by domain. In fashion recommendation, visual features dominate. In book recommendation, text features are more important. Always run an ablation study to understand your domain.

## Exercises

### 🏋️ Exercise 1: Build a Multimodal Recommender with Modality Dropout

Implement modality dropout to make the model robust to missing modalities.

In [ ]:
# TODO: Implement modality dropout
class ModalityDropoutRec(nn.Module):
    """
    TODO:
    1. During training, randomly drop entire modalities (set to zeros)
    2. Each modality is dropped with probability p=0.2
    3. At least one modality must remain
    4. Compare robustness when modalities are missing at test time
    """
    def __init__(self, n_users, text_dim, visual_dim, drop_prob=0.2):
        super().__init__()
        # TODO: Implement
        pass
    
    def forward(self, user_ids, text_feat, visual_feat):
        # TODO: Implement with modality dropout
        pass

# TODO: Evaluate on test set with one modality zeroed out

### 🏋️ Exercise 2: Implement Gated Multimodal Fusion

Build a gating mechanism that dynamically weights modalities per user.

In [ ]:
# TODO: Gated fusion where weights depend on both user and item
# gate = sigmoid(W * [user_embed, text_feat, visual_feat])
# fused = gate * text_encoded + (1 - gate) * visual_encoded
#
# 1. Implement the gating mechanism
# 2. Train and evaluate
# 3. Analyze: which users rely more on visual vs text?
# 4. Visualize gate values across users

### 🏋️ Exercise 3: Audio Feature Integration for Music Recommendation

Add audio features and evaluate the three-modal recommendation system.

In [ ]:
# TODO: Three-modal recommendation (text + visual + audio)
# 1. Use the audio features already generated
# 2. Extend the cross-attention model to handle three modalities
# 3. Compare: 2-modal vs 3-modal performance
# 4. Run ablation: which pair of modalities works best?

## Summary

| Fusion Strategy | Pros | Cons | Best For |
|----------------|------|------|----------|
| **Early Fusion** | Simple, captures raw correlations | Can be dominated by one modality | Small feature dims |
| **Late Fusion** | Modality-specific processing | No cross-modal interaction | Independent modalities |
| **Cross-Attention** | Rich cross-modal interaction | Higher compute cost | Complementary modalities |

**Key Takeaways:**
1. Multimodal features consistently improve recommendation quality
2. Cross-attention fusion captures the richest cross-modal interactions
3. CLIP-style alignment pre-training helps bridge modality gaps
4. Modality importance varies by domain — always run ablation studies
5. Robustness to missing modalities is critical for production systems